In [ ]:
import cv2
import numpy as np
import torch
from tqdm import tqdm
from facenet_pytorch import InceptionResnetV1
import os

# Paths
cfg_path = "C:/Users/justf/Desktop/Sem 6/KP/FaceRecog/yolov4.cfg"
weights_path = "C:/Users/justf/Desktop/Sem 6/KP/FaceRecog/yolov4.weights"

# Load YOLOv4-face
net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

# Load FaceNet
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# Dataset folder
dataset_path = r"C:/Users/justf/Desktop/Sem 6/KP/FaceRecog/Dataset_Face"

embeddings = []
names = []

# ================
# Process dataset
# ================
for person_name in os.listdir(dataset_path):
    person_folder = os.path.join(dataset_path, person_name)
    if not os.path.isdir(person_folder):
        continue

    print(f"/nProcessing person: {person_name}")

    for img_name in tqdm(os.listdir(person_folder)):
        img_path = os.path.join(person_folder, img_name)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Failed to read {img_name}")
            continue

        (h, w) = img.shape[:2]

        # Prepare blob
        blob = cv2.dnn.blobFromImage(img, 1/255.0, (416, 416), swapRB=True, crop=False)
        net.setInput(blob)

        ln = net.getUnconnectedOutLayersNames()
        layer_outputs = net.forward(ln)

        boxes = []
        confidences = []

        for output in layer_outputs:
            for detection in output:
                scores = detection[5:]
                confidence = scores[0]
                if confidence > 0.5:
                    box = detection[0:4] * np.array([w, h, w, h])
                    (centerX, centerY, width, height) = box.astype("int")
                    x = int(centerX - width / 2)
                    y = int(centerY - height / 2)
                    boxes.append([x, y, int(width), int(height)])
                    confidences.append(float(confidence))

        # NMS
        idxs = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.3)

        if len(idxs) == 0:
            print(f"No face detected in {img_name}")
            continue

        # Use first detection
        i = idxs.flatten()[0]
        (x, y) = (boxes[i][0], boxes[i][1])
        (w_box, h_box) = (boxes[i][2], boxes[i][3])

        face_crop = img[y:y+h_box, x:x+w_box]
        if face_crop.size == 0:
            print(f"Empty crop in {img_name}")
            continue

        # Preprocess for FaceNet
        face_rgb = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
        face_resized = cv2.resize(face_rgb, (160, 160))
        face_tensor = torch.tensor(face_resized).permute(2,0,1).float()/255.0
        face_tensor = (face_tensor - 0.5)/0.5
        face_tensor = face_tensor.unsqueeze(0).to(device)

        with torch.no_grad():
            embedding = resnet(face_tensor).cpu().numpy()[0]

        embeddings.append(embedding)
        names.append(person_name)

# Save embeddings & names
np.save("db_embeddings.npy", np.array(embeddings))
np.save("db_names.npy", np.array(names))
print("/n✅ Embedding database created successfully.")

c:\Users\justf\Desktop\Sem 6\KP\FaceRecog\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/nProcessing person: Justin


  1%|          | 3/288 [00:01<01:55,  2.48it/s]

No face detected in Justin_002.jpg


  6%|▌         | 16/288 [00:05<01:14,  3.63it/s]

...

No face detected in Justin_192.jpg


 68%|██████▊   | 195/288 [00:54<00:21,  4.25it/s]

No face detected in Justin_194.jpg


100%|██████████| 288/288 [01:20<00:00,  3.58it/s]

/n✅ Embedding database created successfully.


In [ ]:
import tkinter as tk
from tkinter import simpledialog, messagebox
from PIL import Image, ImageTk
import subprocess
import os
import numpy as np
import torch
from facenet_pytorch import InceptionResnetV1
import joblib
import mediapipe as mp
import time
from sklearn.neighbors import KNeighborsClassifier

# Load FaceNet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

def load_classifier():
    if os.path.exists("face_classifier.pkl"):
        return joblib.load("face_classifier.pkl")
    else:
        return None

clf = load_classifier()

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

def get_ear(landmarks, eye_indices, w, h):
    p = []
    for idx in eye_indices:
        lm = landmarks[idx]
        p.append(np.array([lm.x * w, lm.y * h]))
    A = np.linalg.norm(p[1] - p[5])
    B = np.linalg.norm(p[2] - p[4])
    C = np.linalg.norm(p[0] - p[3])
    return (A + B) / (2.0 * C)

EAR_THRESHOLD = 0.2

root = tk.Tk()
root.title("🔒 AI Face Recognition GUI")

lmain = tk.Label(root)
lmain.pack()

frame_buttons = tk.Frame(root)
frame_buttons.pack()

def add_data():
    global current_embedding
    if current_embedding is None:
        messagebox.showwarning("Peringatan", "Tidak ada wajah terdeteksi.")
        return

    name = simpledialog.askstring("Input Nama", "Masukkan nama orang:")
    if not name:
        return

    # Jalankan collect_dataset.py
    subprocess.run(["python", "collect_dataset.py", name])

    # Load existing embeddings
    if os.path.exists("embeddings.npy"):
        embeddings = np.load("embeddings.npy")
        names = np.load("names.npy")
    else:
        embeddings = np.empty((0,512))
        names = np.array([])

    embeddings = np.vstack([embeddings, current_embedding])
    names = np.append(names, name)

    np.save("embeddings.npy", embeddings)
    np.save("names.npy", names)

    embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
    knn.fit(embeddings_norm, names)
    joblib.dump(knn, "face_classifier.pkl")

    global clf
    clf = knn

    messagebox.showinfo("Info", f"Data '{name}' berhasil ditambahkan dan dataset gambar sudah disimpan.")

btn_add = tk.Button(frame_buttons, text="➕ Tambah Data", command=add_data)
btn_add.pack(side=tk.LEFT, padx=5, pady=5)

cap = cv2.VideoCapture(0)
last_blink_time = time.time()
current_embedding = None

def show_frame():
    global last_blink_time, current_embedding
    ret, frame = cap.read()
    if not ret:
        root.after(10, show_frame)
        return

    frame = cv2.resize(frame, (640,480))
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w = rgb_frame.shape[:2]

    results = face_mesh.process(rgb_frame)

    spoofing = False
    identity = "Unknown"
    confidence = 0.0
    current_embedding = None

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            bbox_x = int(min([lm.x for lm in face_landmarks.landmark]) * w)
            bbox_y = int(min([lm.y for lm in face_landmarks.landmark]) * h)
            bbox_w = int((max([lm.x for lm in face_landmarks.landmark]) - min([lm.x for lm in face_landmarks.landmark])) * w)
            bbox_h = int((max([lm.y for lm in face_landmarks.landmark]) - min([lm.y for lm in face_landmarks.landmark])) * h)

            face_roi = rgb_frame[bbox_y:bbox_y+bbox_h, bbox_x:bbox_x+bbox_w]
            if face_roi.size == 0:
                continue

            face_resized = cv2.resize(face_roi, (160,160))
            face_tensor = torch.tensor(face_resized).permute(2,0,1).float()/255.0
            face_tensor = (face_tensor - 0.5)/0.5
            face_tensor = face_tensor.unsqueeze(0).to(device)

            with torch.no_grad():
                embedding = resnet(face_tensor).cpu().numpy()[0]
                embedding = embedding / np.linalg.norm(embedding)

            current_embedding = embedding

            if clf:
                pred = clf.predict([embedding])[0]
                prob = clf.predict_proba([embedding])[0]
                confidence = np.max(prob)
                if confidence > 0.6:
                    identity = pred

            ear_left = get_ear(face_landmarks.landmark, LEFT_EYE, w, h)
            ear_right = get_ear(face_landmarks.landmark, RIGHT_EYE, w, h)
            avg_ear = (ear_left + ear_right) / 2.0

            if avg_ear < EAR_THRESHOLD:
                last_blink_time = time.time()

            cv2.rectangle(frame, (bbox_x,bbox_y), (bbox_x+bbox_w,bbox_y+bbox_h), (0,255,0), 2)
            cv2.putText(frame, f"{identity} ({confidence:.2f})", (bbox_x, bbox_y-10), cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,255),2)

    if time.time() - last_blink_time > 3:
        spoofing = True
        cv2.putText(frame, "FOTO TERDETEKSI", (20,50), cv2.FONT_HERSHEY_SIMPLEX,1.0,(0,0,255),3)

    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    imgtk = ImageTk.PhotoImage(image=img)
    lmain.imgtk = imgtk
    lmain.configure(image=imgtk)
    root.after(10, show_frame)

show_frame()
root.mainloop()

cap.release()


✅ Camera started. Press 'q' to exit.


In [1]:
import cv2
import numpy as np
import torch
from facenet_pytorch import InceptionResnetV1
from scipy.spatial.distance import cosine
import mediapipe as mp
import time

# =============================
# Load YOLOv4 for face detection
# =============================
cfg_path = "C:/Users/justf/Desktop/Sem 6/KP/FaceRecog/yolov4.cfg"
weights_path = "C:/Users/justf/Desktop/Sem 6/KP/FaceRecog/yolov4.weights"

net = cv2.dnn.readNetFromDarknet(cfg_path, weights_path)
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

# =============================
# Load FaceNet model
# =============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# =============================
# Load embeddings database
# =============================
db_embeddings = np.load("db_embeddings.npy")
db_names = np.load("db_names.npy")
db_embeddings = db_embeddings / np.linalg.norm(db_embeddings, axis=1, keepdims=True)

# =============================
# Mediapipe FaceMesh setup
# =============================
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

# EAR calculation
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

def get_ear(landmarks, eye_indices, w, h):
    p = []
    for idx in eye_indices:
        lm = landmarks[idx]
        p.append(np.array([lm.x * w, lm.y * h]))
    A = np.linalg.norm(p[1] - p[5])
    B = np.linalg.norm(p[2] - p[4])
    C = np.linalg.norm(p[0] - p[3])
    ear = (A + B) / (2.0 * C)
    return ear

EAR_THRESHOLD = 0.2

# =============================
# Video Capture
# =============================
cap = cv2.VideoCapture(0)
print("✅ Camera started. Press 'q' to exit.")

last_blink_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (800, 600))
    h, w = frame.shape[:2]

    # === YOLO detection ===
    blob = cv2.dnn.blobFromImage(frame, 1/255.0, (416,416), swapRB=True, crop=False)
    net.setInput(blob)
    ln = net.getUnconnectedOutLayersNames()
    outputs = net.forward(ln)

    boxes = []
    confidences = []

    for output in outputs:
        for detection in output:
            scores = detection[5:]
            confidence = scores[0]
            if confidence > 0.5:
                box = detection[0:4] * np.array([w, h, w, h])
                centerX, centerY, width, height = box.astype("int")

                # === Kotak deteksi===
                scale = 0.8  # MODIFIED
                width = int(width * scale)
                height = int(height * scale)
                x = int(centerX - width/2)
                y = int(centerY - height/2)

                boxes.append([x, y, width, height])
                confidences.append(float(confidence))

    idxs = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.3)

    spoofing = False

    if len(idxs) > 0:
        for i in idxs.flatten():
            x, y, w_box, h_box = boxes[i]
            cv2.rectangle(frame, (x,y), (x+w_box,y+h_box), (0,255,0), 2)

            face_roi = frame[y:y+h_box, x:x+w_box]
            if face_roi.size == 0:
                continue

            # === FaceNet Recognition ===
            face_rgb_resized = cv2.resize(cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB), (160,160))
            face_tensor = torch.tensor(face_rgb_resized).permute(2,0,1).float()/255.0
            face_tensor = (face_tensor - 0.5)/0.5
            face_tensor = face_tensor.unsqueeze(0).to(device)

            with torch.no_grad():
                embedding = resnet(face_tensor).cpu().numpy()[0]
                embedding = embedding / np.linalg.norm(embedding)

            max_similarity = -1
            identity = "Unknown"
            for db_emb, name in zip(db_embeddings, db_names):
                similarity = 1 - cosine(embedding, db_emb)
                if similarity > max_similarity:
                    max_similarity = similarity
                    identity = name
            if max_similarity < 0.6:
                identity = "Unknown"

            # === FaceMesh EAR detection ===
            rgb_face = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_face)

            if results.multi_face_landmarks:
                for face_landmarks in results.multi_face_landmarks:
                    ear_left = get_ear(face_landmarks.landmark, LEFT_EYE, w_box, h_box)
                    ear_right = get_ear(face_landmarks.landmark, RIGHT_EYE, w_box, h_box)
                    avg_ear = (ear_left + ear_right) / 2.0

                    if avg_ear < EAR_THRESHOLD:
                        last_blink_time = time.time()

            cv2.putText(
                frame,
                f"{identity} ({max_similarity:.2f})",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255,255,0),
                2
            )

    # If >3 seconds no blink
    if time.time() - last_blink_time > 3:
        spoofing = True
        cv2.putText(frame, "FOTO TERDETEKSI", (30,50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,0,255),3)

    cv2.imshow("AI Door Lock System", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


c:\Users\justf\Desktop\Sem 6\KP\FaceRecog\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Camera started. Press 'q' to exit.


In [3]:
# train_classifier.py
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
import joblib

# Load embeddings dan label
db_embeddings = np.load("db_embeddings.npy")
db_names = np.load("db_names.npy")

# Normalisasi embeddings
db_embeddings = db_embeddings / np.linalg.norm(db_embeddings, axis=1, keepdims=True)

# Latih classifier
clf = KNeighborsClassifier(n_neighbors=1, metric="cosine")
clf.fit(db_embeddings, db_names)

# Simpan ke pkl
joblib.dump(clf, "face_classifier.pkl")
print("✅ Classifier berhasil disimpan ke 'face_classifier.pkl'")


✅ Classifier berhasil disimpan ke 'face_classifier.pkl'


In [7]:
import tkinter as tk
from tkinter import simpledialog, messagebox
from PIL import Image, ImageTk
import cv2
import numpy as np
import torch
from facenet_pytorch import InceptionResnetV1
import joblib
import mediapipe as mp
import time
import os
from sklearn.neighbors import KNeighborsClassifier

# Load model FaceNet
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

# Load or initialize classifier
def load_classifier():
    if os.path.exists("face_classifier.pkl"):
        return joblib.load("face_classifier.pkl")
    else:
        return None

clf = load_classifier()

# Mediapipe FaceMesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

# EAR calculation
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

def get_ear(landmarks, eye_indices, w, h):
    p = []
    for idx in eye_indices:
        lm = landmarks[idx]
        p.append(np.array([lm.x * w, lm.y * h]))
    A = np.linalg.norm(p[1] - p[5])
    B = np.linalg.norm(p[2] - p[4])
    C = np.linalg.norm(p[0] - p[3])
    ear = (A + B) / (2.0 * C)
    return ear

EAR_THRESHOLD = 0.2

# Tkinter Window
root = tk.Tk()
root.title("🔒 AI Face Recognition GUI")

lmain = tk.Label(root)
lmain.pack()

# Buttons
frame_buttons = tk.Frame(root)
frame_buttons.pack()

def add_data():
    global current_embedding
    if current_embedding is None:
        messagebox.showwarning("Peringatan", "Tidak ada wajah terdeteksi.")
        return

    name = simpledialog.askstring("Input Nama", "Masukkan nama orang:")
    if not name:
        return

    # Load existing embeddings
    if os.path.exists("embeddings.npy"):
        embeddings = np.load("embeddings.npy")
        names = np.load("names.npy")
    else:
        embeddings = np.empty((0,512))
        names = np.array([])

    # Append new embedding
    embeddings = np.vstack([embeddings, current_embedding])
    names = np.append(names, name)

    # Save updated data
    np.save("embeddings.npy", embeddings)
    np.save("names.npy", names)

    # Retrain classifier
    embeddings_norm = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    knn = KNeighborsClassifier(n_neighbors=1, metric="cosine")
    knn.fit(embeddings_norm, names)
    joblib.dump(knn, "face_classifier.pkl")

    # Reload classifier
    global clf
    clf = knn

    messagebox.showinfo("Info", f"Data '{name}' berhasil ditambahkan dan model diperbarui.")

btn_add = tk.Button(frame_buttons, text="➕ Tambah Data", command=add_data)
btn_add.pack(side=tk.LEFT, padx=5, pady=5)

cap = cv2.VideoCapture(0)
last_blink_time = time.time()
current_embedding = None  # Untuk menyimpan embedding saat ini

def show_frame():
    global last_blink_time, current_embedding

    ret, frame = cap.read()
    if not ret:
        root.after(10, show_frame)
        return

    frame = cv2.resize(frame, (640,480))
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w = rgb_frame.shape[:2]

    results = face_mesh.process(rgb_frame)

    spoofing = False
    identity = "Unknown"
    confidence = 0.0
    current_embedding = None

    if results.multi_face_landmarks:
        for face_landmarks in results.multi_face_landmarks:
            bbox_x = int(min([lm.x for lm in face_landmarks.landmark]) * w)
            bbox_y = int(min([lm.y for lm in face_landmarks.landmark]) * h)
            bbox_w = int((max([lm.x for lm in face_landmarks.landmark]) - min([lm.x for lm in face_landmarks.landmark])) * w)
            bbox_h = int((max([lm.y for lm in face_landmarks.landmark]) - min([lm.y for lm in face_landmarks.landmark])) * h)

            face_roi = rgb_frame[bbox_y:bbox_y+bbox_h, bbox_x:bbox_x+bbox_w]
            if face_roi.size == 0:
                continue

            face_resized = cv2.resize(face_roi, (160,160))
            face_tensor = torch.tensor(face_resized).permute(2,0,1).float()/255.0
            face_tensor = (face_tensor - 0.5)/0.5
            face_tensor = face_tensor.unsqueeze(0).to(device)

            with torch.no_grad():
                embedding = resnet(face_tensor).cpu().numpy()[0]
                embedding = embedding / np.linalg.norm(embedding)

            current_embedding = embedding  # simpan embedding saat ini

            if clf:
                pred = clf.predict([embedding])[0]
                prob = clf.predict_proba([embedding])[0]
                confidence = np.max(prob)
                if confidence > 0.6:
                    identity = pred

            ear_left = get_ear(face_landmarks.landmark, LEFT_EYE, w, h)
            ear_right = get_ear(face_landmarks.landmark, RIGHT_EYE, w, h)
            avg_ear = (ear_left + ear_right) / 2.0

            if avg_ear < EAR_THRESHOLD:
                last_blink_time = time.time()

            cv2.rectangle(frame, (bbox_x,bbox_y), (bbox_x+bbox_w,bbox_y+bbox_h), (0,255,0), 2)
            cv2.putText(frame, f"{identity} ({confidence:.2f})", (bbox_x, bbox_y-10), cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,255),2)

    if time.time() - last_blink_time > 3:
        spoofing = True
        cv2.putText(frame, "FOTO TERDETEKSI", (20,50), cv2.FONT_HERSHEY_SIMPLEX,1.0,(0,0,255),3)

    img = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    imgtk = ImageTk.PhotoImage(image=img)
    lmain.imgtk = imgtk
    lmain.configure(image=imgtk)
    root.after(10, show_frame)

show_frame()
root.mainloop()

cap.release()
